In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import shutil


TRAIN_IMG = "/content/drive/MyDrive/3_classess_yolo_new/train/3_clasess_labeled image_yolo"
TRAIN_LBL = "/content/drive/MyDrive/3_classess_yolo_new/train/yolo1.1_3_classess_dataset/obj_train_data"

VAL_IMG = "/content/drive/MyDrive/3_classess_yolo_new/validation/Val_images"
VAL_LBL = "/content/drive/MyDrive/3_classess_yolo_new/validation/Val_txt"

TEST_IMG = "/content/drive/MyDrive/3_classess_yolo_new/test/test_3_clasess_images"
TEST_LBL = "/content/drive/MyDrive/3_classess_yolo_new/test/test_3_clasess/obj_train_data"

OUT = "/content/drive/MyDrive/3_classess_yolo_new/Results"

# ===================== CREATE YOLO FOLDER STRUCTURE =====================
YOLO_BASE = "/content/drive/MyDrive/3_classess_yolo_new"

# Create folders
folders = [
    f"{YOLO_BASE}/images/train",
    f"{YOLO_BASE}/images/val",
    f"{YOLO_BASE}/images/test",
    f"{YOLO_BASE}/labels/train",
    f"{YOLO_BASE}/labels/val",
    f"{YOLO_BASE}/labels/test"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# ===================== ORGANIZE FILES =====================
print("📂 Organizing dataset files...")

def copy_files(img_src, lbl_src, img_dest, lbl_dest):
    """Copy images and labels to YOLO structure"""
    img_count = 0
    lbl_count = 0

    # Copy images
    if os.path.exists(img_src):
        for file in os.listdir(img_src):
            if file.endswith(('.jpg', '.jpeg', '.png')):
                src = os.path.join(img_src, file)
                dst = os.path.join(img_dest, file)
                if not os.path.exists(dst):
                    shutil.copy(src, dst)
                img_count += 1
    else:
        print(f"⚠️ Warning: {img_src} does not exist!")

    # Copy labels
    if os.path.exists(lbl_src):
        for file in os.listdir(lbl_src):
            if file.endswith('.txt'):
                src = os.path.join(lbl_src, file)
                dst = os.path.join(lbl_dest, file)
                if not os.path.exists(dst):
                    shutil.copy(src, dst)
                lbl_count += 1
    else:
        print(f"⚠️ Warning: {lbl_src} does not exist!")

    return img_count, lbl_count

# Organize train files
train_imgs, train_lbls = copy_files(
    TRAIN_IMG, TRAIN_LBL,
    f"{YOLO_BASE}/images/train",
    f"{YOLO_BASE}/labels/train"
)

# Organize validation files
val_imgs, val_lbls = copy_files(
    VAL_IMG, VAL_LBL,
    f"{YOLO_BASE}/images/val",
    f"{YOLO_BASE}/labels/val"
)

# Organize test files
test_imgs, test_lbls = copy_files(
    TEST_IMG, TEST_LBL,
    f"{YOLO_BASE}/images/test",
    f"{YOLO_BASE}/labels/test"
)

print(f"\n✅ Files organized:")
print(f"   Train:      {train_imgs} images, {train_lbls} labels")
print(f"   Validation: {val_imgs} images, {val_lbls} labels")
print(f"   Test:       {test_imgs} images, {test_lbls} labels")

# ===================== CREATE DATA.YAML =====================
yaml_content = """path: /content/drive/MyDrive/3_classess_yolo_new
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: Speed_Breaker
  1: Slippery_Road
  2: School_Zone
"""

os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ data.yaml created")

In [ ]:
# ===================== INSTALL & IMPORT =====================
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO

print("✅ YOLO imported successfully")

In [ ]:
# ===================== TRAIN MODEL =====================
model = YOLO("yolov8n.pt")

print("\n" + "="*80)
print("🚀 STARTING TRAINING - Metrics will be printed after each epoch")
print("="*80 + "\n")

results = model.train(
    data=f"{OUT}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project=OUT,
    name="yolo_training",
    plots=True,
    save=True,
    verbose=True  # This prints metrics after each epoch
)

print("\n✅ Training completed!")

In [ ]:
# ===================== PRINT DETAILED EPOCH-BY-EPOCH METRICS =====================
print("\n" + "="*80)



results_csv = "/content/drive/MyDrive/3_classess_yolo_new/Results/yolo_training4/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("\n{:<8} {:<15} {:<15} {:<15} {:<15} {:<15} {:<15}".format(
    "Epoch", "Train Loss", "Val Loss", "Val mAP50", "Val mAP50-95", "Precision", "Recall"
))
print("-" * 110)

for idx, row in df.iterrows():
    print("{:<8} {:<15.4f} {:<15.4f} {:<15.4f} {:<15.4f} {:<15.4f} {:<15.4f}".format(
        int(row['epoch']),
        row['train/box_loss'],
        row['val/box_loss'],
        row['metrics/mAP50(B)'],
        row['metrics/mAP50-95(B)'],
        row['metrics/precision(B)'],
        row['metrics/recall(B)']
    ))

# ===================== TEST ON TEST SET =====================
print("\n" + "="*80)
print("🧪 EVALUATING ON TEST SET")
print("="*80 + "\n")

best_model = YOLO(f"{OUT}/yolo_training4/weights/best.pt")
test_results = best_model.val(data=f"{OUT}/data.yaml", split="test")

print(f"\n🎯 FINAL TEST RESULTS:")
print(f"   mAP50:        {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"   mAP50-95:     {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"   Precision:    {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"   Recall:       {test_results.results_dict['metrics/recall(B)']:.4f}")

In [ ]:
# ===================== TEST ON TEST SET =====================
print("\n" + "="*80)
print("🧪 EVALUATING ON TEST SET")
print("="*80 + "\n")

best_model = YOLO(f"{OUT}/yolo_training4/weights/best.pt")
test_results = best_model.val(data=f"{OUT}/data.yaml", split="test")

print(f"\n🎯 FINAL TEST RESULTS:")
print(f"   mAP50:        {test_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"   mAP50-95:     {test_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"   Precision:    {test_results.results_dict['metrics/precision(B)']:.4f}")
print(f"   Recall:       {test_results.results_dict['metrics/recall(B)']:.4f}")

In [ ]:
# ===================== PLOT TRAINING CURVES =====================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: Training & Validation Box Loss
axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Loss', marker='o', linewidth=2, color='blue')
axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Loss', marker='s', linewidth=2, color='orange')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Box Loss', fontsize=12)
axes[0, 0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Validation mAP50 (Main Accuracy Metric)
axes[0, 1].plot(df['epoch'], df['metrics/mAP50(B)'], label='Val mAP50', marker='o', color='green', linewidth=2)
axes[0, 1].axhline(y=test_results.results_dict['metrics/mAP50(B)'], color='red', linestyle='--', label=f'Test mAP50: {test_results.results_dict["metrics/mAP50(B)"]:.3f}')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('mAP50', fontsize=12)
axes[0, 1].set_title('Validation Accuracy (mAP50) vs Epoch', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Validation mAP50-95
axes[0, 2].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Val mAP50-95', marker='o', color='purple', linewidth=2)
axes[0, 2].axhline(y=test_results.results_dict['metrics/mAP50-95(B)'], color='red', linestyle='--', label=f'Test mAP50-95: {test_results.results_dict["metrics/mAP50-95(B)"]:.3f}')
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('mAP50-95', fontsize=12)
axes[0, 2].set_title('Validation mAP50-95 vs Epoch', fontsize=14, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Precision
axes[1, 0].plot(df['epoch'], df['metrics/precision(B)'], label='Val Precision', marker='o', color='teal', linewidth=2)
axes[1, 0].axhline(y=test_results.results_dict['metrics/precision(B)'], color='red', linestyle='--', label=f'Test Precision: {test_results.results_dict["metrics/precision(B)"]:.3f}')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Precision', fontsize=12)
axes[1, 0].set_title('Validation Precision vs Epoch', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Recall
axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Val Recall', marker='s', color='brown', linewidth=2)
axes[1, 1].axhline(y=test_results.results_dict['metrics/recall(B)'], color='red', linestyle='--', label=f'Test Recall: {test_results.results_dict["metrics/recall(B)"]:.3f}')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Recall', fontsize=12)
axes[1, 1].set_title('Validation Recall vs Epoch', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: All Training Losses
axes[1, 2].plot(df['epoch'], df['train/box_loss'], label='Box Loss', marker='o', linewidth=2)
axes[1, 2].plot(df['epoch'], df['train/cls_loss'], label='Class Loss', marker='s', linewidth=2)
axes[1, 2].plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss', marker='^', linewidth=2)
axes[1, 2].set_xlabel('Epoch', fontsize=12)
axes[1, 2].set_ylabel('Loss', fontsize=12)
axes[1, 2].set_title('All Training Losses vs Epoch', fontsize=14, fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/training_curves.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Plots saved to {OUT}/training_curves.png")